In [ ]:
import pandas as pd
import numpy as np
from hierarchicalforecast.utils import aggregate


In [ ]:
# ---------------------------------------------------------------
# Load from existing saved files (NOT recreating — reading what
# already exists on disk, per Pool A's actual environment pattern)
# ---------------------------------------------------------------
Y_product = pd.read_csv("../eabl parsed csv/Y_product.csv")
Y_product["ds"] = pd.to_datetime(Y_product["ds"])
S_product = pd.read_csv("../eabl parsed csv/S_product.csv", index_col=0)

# Raw inspection first, before any checks
print(Y_product.head())
print("Y_product shape:", Y_product.shape)


In [ ]:
print(S_product.head())
print("S_product shape:", S_product.shape)

In [ ]:
results = {}  # collects True/False per check for the final summary

# ---------------------------------------------------------------
# 1 & 2. Column names check
# ---------------------------------------------------------------
expected_cols = ["unique_id", "ds", "y"]
actual_cols = list(Y_product.columns)
actual_cols


In [ ]:
results["columns_match"] = set(expected_cols).issubset(actual_cols)
results["columns_match"]

In [ ]:
print("Y_product columns:", actual_cols, "-> expected at least:", expected_cols)

In [ ]:
# ---------------------------------------------------------------
# 3. Dtype checks: ds must be datetime, y must be numeric
# ---------------------------------------------------------------
results["ds_is_datetime"] = pd.api.types.is_datetime64_any_dtype(Y_product["ds"])
results["ds_is_datetime"]

In [ ]:
results["y_is_numeric"] = pd.api.types.is_numeric_dtype(Y_product["y"])
results["y_is_numeric"]

In [ ]:

print("ds dtype:", Y_product["ds"].dtype, "| y dtype:", Y_product["y"].dtype)

In [ ]:
# ---------------------------------------------------------------
# 4. Every ds value must fall on a Monday (weekday() == 0)
# ---------------------------------------------------------------
non_monday = Y_product.loc[Y_product["ds"].dt.weekday != 0]
non_monday

In [ ]:
results["all_dates_are_monday"] = len(non_monday) == 0
results["all_dates_are_monday"]

In [ ]:
print("Non-Monday rows found:", len(non_monday))

In [ ]:
# ---------------------------------------------------------------
# 5. No duplicate (unique_id, ds) pairs
# ---------------------------------------------------------------
dupe_count = Y_product.duplicated(subset=["unique_id", "ds"]).sum()
dupe_count

In [ ]:
results["no_duplicate_pairs"] = dupe_count == 0
results["no_duplicate_pairs"]

In [ ]:
print("Duplicate (unique_id, ds) pairs:", dupe_count)
# Pool A's variant used: assert dupe_count == 0, "Duplicate rows found..."
# Equivalent check, fail-fast style — see merge log for why only the
# summary-dict version is kept as the primary form here.

In [ ]:

# ---------------------------------------------------------------
# 6. Inspect S_product's structure before assuming anything about it
# ---------------------------------------------------------------
print("\nS_product type:", type(S_product))


In [ ]:
print("S_product index sample:", list(S_product.index[:5]))


In [ ]:
print("S_product columns sample:", list(S_product.columns[:5]))

In [ ]:
S_product.columns

In [ ]:
len(S_product["unique_id"])

In [ ]:
if "unique_id" in S_product.columns:
    bottom_ids_from_S = set(S_product["unique_id"])
    print(f"The columns are {bottom_ids_from_S }\n")
    print(f"The length is {len(bottom_ids_from_S)})
else:
    bottom_ids_from_S = set(S_product.columns)
    



In [ ]:
n_bottom = len(bottom_ids_from_S)
n_bottom

In [ ]:
results["bottom_level_count_is_222"] = n_bottom == 222
results["bottom_level_count_is_222"] 


In [ ]:
print("Bottom-level count implied by S_product:", n_bottom, "(expected 222)")

In [ ]:
print(S_product.shape)

In [ ]:
print(S_product.head())


In [ ]:
print(S_product.columns.tolist()[:10])


In [ ]:

# ---------------------------------------------------------------
# 7 & 8. Y_product vs S_product alignment
# ---------------------------------------------------------------
y_product_ids = set(Y_product["unique_id"].unique())
print("\nTotal distinct unique_ids in Y_product (all hierarchy levels):", len(y_product_ids))
print("Distinct series in Y_product (nunique):", Y_product["unique_id"].nunique())

missing_from_Y = bottom_ids_from_S - y_product_ids
results["Y_product_matches_S_bottom"] = len(missing_from_Y) == 0
print("Bottom-level ids missing from Y_product:", len(missing_from_Y))

In [ ]:


# ---------------------------------------------------------------
# 9a. Full per-series gap report — all series, not just one
# ---------------------------------------------------------------
gap_report = []
for uid, group in Y_product.groupby("unique_id"):
    start, end = group["ds"].min(), group["ds"].max()
    expected_weeks = pd.date_range(start, end, freq="W-MON")
    observed_weeks = set(group["ds"])
    missing_weeks = len(set(expected_weeks) - observed_weeks)
    zero_weeks = (group["y"] == 0).sum()
    gap_report.append({
        "unique_id": uid,
        "n_expected_weeks": len(expected_weeks),
        "n_observed_rows": len(group),
        "missing_weeks": missing_weeks,
        "zero_sale_weeks": zero_weeks,
    })
gap_report = pd.DataFrame(gap_report)

total_missing_weeks = gap_report["missing_weeks"].sum()
total_zero_weeks = gap_report["zero_sale_weeks"].sum()
print(f"\nTotal true gap-weeks across all series (NOT filled): {total_missing_weeks}")
print(f"Total recorded zero-sale weeks across all series: {total_zero_weeks}")
print(gap_report.sort_values("missing_weeks", ascending=False).head(5))

In [ ]:


# ---------------------------------------------------------------
# 9b. Quick human-readable spot-check on ONE series
# (kept from Pool A's variant — simpler to read than the full table)
# ---------------------------------------------------------------
full_calendar = pd.date_range(Y_product["ds"].min(), Y_product["ds"].max(), freq="W-MON")
sample_id = Y_product["unique_id"].iloc[0]
sample_series = (
    Y_product[Y_product["unique_id"] == sample_id]
    .set_index("ds")["y"]
    .reindex(full_calendar)
)
print(f"\nSample series: {sample_id}")
print("True gaps (no row at all):", sample_series.isna().sum())
print("Recorded zeros:", (sample_series == 0).sum())
print("Recorded nonzero sales:", (sample_series > 0).sum())

In [ ]:


# ---------------------------------------------------------------
# 10. No NaN values in y
# ---------------------------------------------------------------
nan_count = Y_product["y"].isna().sum()
results["no_nan_in_y"] = nan_count == 0
print("\nNaN values in y:", nan_count)

In [ ]:


# ---------------------------------------------------------------
# 11. Date range / series length summary
# ---------------------------------------------------------------
print("\nOverall ds range:", Y_product["ds"].min(), "to", Y_product["ds"].max())
print(Y_product.groupby("unique_id").size().describe())


In [ ]:

# ---------------------------------------------------------------
# Final pass/fail summary
# ---------------------------------------------------------------
print("\n=== STEP 2.1 VALIDATION SUMMARY ===")
for check, passed in results.items():
    print(f"{'PASS' if passed else 'FAIL'} — {check}")

overall = all(results.values())
print(f"\nOVERALL: {'PASS' if overall else 'FAIL'}")
print("Y_product is ready for StatsForecast:", not Y_product[["unique_id", "ds", "y"]].isna().any().any())
print("Reminder: gaps and zero-sale weeks were only counted and reported above — nothing was filled or imputed.")
=

### What each block does

- **Column check** — confirms the exact three columns StatsForecast requires exist; stops immediately with a clear error if not, rather than failing confusingly later inside the library.
- **Date check** — confirms `ds` is a real date type (not text) and shows the actual span of history you're working with.
- **Frequency check** — rebuilds the expected weekly calendar so you can compare it against what's actually in `Y_product` later if needed.
- **Duplicate check** — this is critical and easy to miss: if the same `unique_id`/`ds` pair appears twice, StatsForecast can silently misbehave, since it expects exactly one value per series per time point.
- **Missing-vs-zero check** — reuses the exact logic from your Step 4.1 work to confirm, on a real example, that gaps and zeros are genuinely distinguishable in this table — not just assumed to be.
- **Final readiness check** — one clean `True`/`False` answer: is there any missing value sitting in the three required columns themselves (not the overall series, just these specific columns), which would break fitting outright.

In [ ]:
# ---------- Confirm the required 3-column structure ----------
print(Y_product.columns.tolist())
assert set(["unique_id", "ds", "y"]).issubset(Y_product.columns), "Missing required columns"

# ---------- Confirm ds is a proper datetime, and check the date range ----------
Y_product["ds"] = pd.to_datetime(Y_product["ds"])
print("Date range:", Y_product["ds"].min(), "to", Y_product["ds"].max())

# ---------- Confirm the frequency is consistently weekly, Monday-anchored ----------
full_calendar = pd.date_range(Y_product["ds"].min(), Y_product["ds"].max(), freq="W-MON")
print("Expected weeks:", len(full_calendar))

# ---------- Confirm no duplicate (unique_id, ds) pairs ----------
dupes = Y_product.duplicated(subset=["unique_id", "ds"], keep=False)
print("Duplicate (unique_id, ds) rows:", dupes.sum())
assert dupes.sum() == 0, "Duplicate rows found — StatsForecast requires one row per series per date"

# ---------- Confirm missing-vs-zero distinction is real, not accidental ----------
# For one sample series, show whether gaps and zeros look genuinely different
sample_id = Y_product["unique_id"].iloc[0]
sample_series = (
    Y_product[Y_product["unique_id"] == sample_id]
    .set_index("ds")["y"]
    .reindex(full_calendar)
)
print(f"\nSample series: {sample_id}")
print("True gaps (no row at all):", sample_series.isna().sum())
print("Recorded zeros:", (sample_series == 0).sum())
print("Recorded nonzero sales:", (sample_series > 0).sum())

# ---------- Final readiness check ----------
print("\nY_product is ready for StatsForecast:", not Y_product[["unique_id", "ds", "y"]].isna().any().any())


In [ ]:
<role>
You are a technical editor merging two overlapping documents about the same forecasting project into one authoritative reference. Accuracy and completeness matter more than brevity.
</role>

<context>
Both pools describe Task F1, Hierarchy A of a demand-forecasting project:
- Weekly data (freq="W-MON"), 222 SKUs, ~70 weeks of history, 13-week horizon
- SKU mix: 102 smooth, 39 erratic, 20 intermittent, 61 lumpy
- Reference text: FPP3 (Hyndman & Athanasopoulos, otexts.com/fpp3)
- Library: Nixtla StatsForecast (+ hierarchicalforecast for reconciliation)

Pool A is the primary guide. It contains:
- a sourcing note
- a full model inventory (Part 1)
- a step-by-step guide (2.1–2.6) with code, a checklist and pitfalls

Pool B is a derived analysis of Pool A. It contains:
- two step-to-checklist mapping tables (one simple, one with "spans" notes)
- a flagged gap about winning-model selection
- the candidate models regrouped by role (Baselines, Exponential smoothing, ARIMA, Intermittent, Decomposition, Theta, Outside StatsForecast)
- a note excluding reconciliation from Step 2.3, because it belongs to task F3
</context>

<pool_a>
mport pandas as pd
import numpy as np

results = {}  # collects True/False per check for the final summary

# ---------------------------------------------------------------
# 1 & 2. Column names check
# ---------------------------------------------------------------
expected_cols = ["unique_id", "ds", "y"]
actual_cols = list(Y_product.columns)
results["columns_match"] = actual_cols == expected_cols
print("Y_product columns:", actual_cols, "-> expected:", expected_cols)

# ---------------------------------------------------------------
# 3. Dtype checks: ds must be datetime, y must be numeric
# ---------------------------------------------------------------
results["ds_is_datetime"] = pd.api.types.is_datetime64_any_dtype(Y_product["ds"])
results["y_is_numeric"] = pd.api.types.is_numeric_dtype(Y_product["y"])
print("ds dtype:", Y_product["ds"].dtype, "| y dtype:", Y_product["y"].dtype)

# ---------------------------------------------------------------
# 4. Every ds value must fall on a Monday (weekday() == 0)
# ---------------------------------------------------------------
non_monday = Y_product.loc[Y_product["ds"].dt.weekday != 0]
results["all_dates_are_monday"] = len(non_monday) == 0
print("Non-Monday rows found:", len(non_monday))

# ---------------------------------------------------------------
# 5. No duplicate (unique_id, ds) pairs
# ---------------------------------------------------------------
dupe_count = Y_product.duplicated(subset=["unique_id", "ds"]).sum()
results["no_duplicate_pairs"] = dupe_count == 0
print("Duplicate (unique_id, ds) pairs:", dupe_count)

# ---------------------------------------------------------------
# 6. Inspect S_product's structure before assuming anything about it
#    (the knowledge pool does not specify its exact shape)
# ---------------------------------------------------------------
print("\nS_product type:", type(S_product))
print("S_product index sample:", list(S_product.index[:5]))
print("S_product columns sample:", list(S_product.columns[:5]))

# Determine where the bottom-level identifiers live: if "unique_id" is a
# column, use it; otherwise assume the row/column index holds them, and
# the bottom-level count equals the number of columns (per the earlier
# established convention: S's columns = bottom-level series).
if "unique_id" in S_product.columns:
    bottom_ids_from_S = set(S_product["unique_id"])
else:
    bottom_ids_from_S = set(S_product.columns)

n_bottom = len(bottom_ids_from_S)
results["bottom_level_count_is_222"] = n_bottom == 222
print("Bottom-level count implied by S_product:", n_bottom, "(expected 222)")

# ---------------------------------------------------------------
# 7 & 8. Total unique_ids in Y_product, and whether S_product's
#    bottom-level ids are all present inside Y_product
# ---------------------------------------------------------------
y_product_ids = set(Y_product["unique_id"].unique())
print("\nTotal distinct unique_ids in Y_product (all hierarchy levels):", len(y_product_ids))
print("NOTE: this total includes Total/Category/Brand rollups, not just the 222 bottom SKUs.")

missing_from_Y = bottom_ids_from_S - y_product_ids
results["Y_product_matches_S_bottom"] = len(missing_from_Y) == 0
print("Bottom-level ids missing from Y_product:", len(missing_from_Y))

# ---------------------------------------------------------------
# 9. Per-series calendar check: gaps vs. zero-sale weeks, kept SEPARATE
#    and NOT filled or imputed, per FPP3 §13.9 and your Step 4.1 decision
# ---------------------------------------------------------------
gap_report = []
for uid, group in Y_product.groupby("unique_id"):
    start, end = group["ds"].min(), group["ds"].max()
    expected_weeks = pd.date_range(start, end, freq="W-MON")
    observed_weeks = set(group["ds"])
    missing_weeks = len(set(expected_weeks) - observed_weeks)     # true gaps — no row at all
    zero_weeks = (group["y"] == 0).sum()                          # recorded zeros — a real observation
    gap_report.append({
        "unique_id": uid,
        "n_expected_weeks": len(expected_weeks),
        "n_observed_rows": len(group),
        "missing_weeks": missing_weeks,
        "zero_sale_weeks": zero_weeks,
    })
gap_report = pd.DataFrame(gap_report)

total_missing_weeks = gap_report["missing_weeks"].sum()
total_zero_weeks = gap_report["zero_sale_weeks"].sum()
print(f"\nTotal true gap-weeks across all series (NOT filled): {total_missing_weeks}")
print(f"Total recorded zero-sale weeks across all series: {total_zero_weeks}")
print("Series with the most gaps (top 5):")
print(gap_report.sort_values("missing_weeks", ascending=False).head(5))

# ---------------------------------------------------------------
# 10. No NaN values in y
# ---------------------------------------------------------------
nan_count = Y_product["y"].isna().sum()
results["no_nan_in_y"] = nan_count == 0
print("\nNaN values in y:", nan_count)

# ---------------------------------------------------------------
# 11. Date range / series length summary
# ---------------------------------------------------------------
print("\nOverall ds range:", Y_product["ds"].min(), "to", Y_product["ds"].max())
print("Per-series row-count summary:")
print(Y_product.groupby("unique_id").size().describe())

# ---------------------------------------------------------------
# Final pass/fail summary
# ---------------------------------------------------------------
print("\n=== STEP 2.1 VALIDATION SUMMARY ===")
for check, passed in results.items():
    print(f"{'PASS' if passed else 'FAIL'} — {check}")

overall = all(results.values())
print(f"\nOVERALL: {'PASS' if overall else 'FAIL'}")
print("Reminder: gaps and zero-sale weeks were only counted and reported above — nothing was filled or imputed.")
and the following code:Y_product = pd.read_csv("../eabl parsed csv/Y_product.csv")
print(Y_product)
Y_product["ds"] = pd.to_datetime(Y_product["ds"])

print(Y_product)
S_product = pd.read_csv("../eabl parsed csv/S_product.csv", index_col=0)print(S_product)S_product.shape# ---------- Basic checks before trusting the data ----------
print("Y_product shape:", Y_product.shape)print("Y_product columns:", Y_product.columns.tolist())
print("Distinct series in Y_product:", Y_product["unique_id"].nunique())print("Date range:", Y_product["ds"].min(), "to", Y_product["ds"].max())print("S_product shape:", S_product.shape)
print(Y_product.head())

# ---------- Confirm the required 3-column structure ----------
assert set(["unique_id", "ds", "y"]).issubset(Y_product.columns), "Missing required columns"


# ---------- Confirm the frequency is consistently weekly, Monday-anchored ----------
full_calendar = pd.date_range(Y_product["ds"].min(), Y_product["ds"].max(), freq="W-MON")
print("Expected weeks:", len(full_calendar))

# ---------- Confirm no duplicate (unique_id, ds) pairs ----------
dupes = Y_product.duplicated(subset=["unique_id", "ds"], keep=False)
print("Duplicate (unique_id, ds) rows:", dupes.sum())
assert dupes.sum() == 0, "Duplicate rows found — StatsForecast requires one row per series per date"

# ---------- Confirm missing-vs-zero distinction is real, not accidental ----------
sample_id = Y_product["unique_id"].iloc[0]
sample_series = (
    Y_product[Y_product["unique_id"] == sample_id]
    .set_index("ds")["y"]
    .reindex(full_calendar)
)
print(f"\nSample series: {sample_id}")
print("True gaps (no row at all):", sample_series.isna().sum())
print("Recorded zeros:", (sample_series == 0).sum())
print("Recorded nonzero sales:", (sample_series > 0).sum())

# ---------- Final readiness check ----------
print("\nY_product is ready for StatsForecast:", not Y_product[["unique_id", "ds", "y"]].isna().any().any())
print("Shape:", Y_product.shape, "  S_product shape:", S_product.shape)
# ---------- Confirm the required 3-column structure ----------
print(Y_product.columns.tolist())
assert set(["unique_id", "ds", "y"]).issubset(Y_product.columns), "Missing required columns"

# ---------- Confirm ds is a proper datetime, and check the date range ----------
Y_product["ds"] = pd.to_datetime(Y_product["ds"])
print("Date range:", Y_product["ds"].min(), "to", Y_product["ds"].max())

# ---------- Confirm the frequency is consistently weekly, Monday-anchored ----------
full_calendar = pd.date_range(Y_product["ds"].min(), Y_product["ds"].max(), freq="W-MON")
print("Expected weeks:", len(full_calendar))

# ---------- Confirm no duplicate (unique_id, ds) pairs ----------
dupes = Y_product.duplicated(subset=["unique_id", "ds"], keep=False)
print("Duplicate (unique_id, ds) rows:", dupes.sum())
assert dupes.sum() == 0, "Duplicate rows found — StatsForecast requires one row per series per date"

# ---------- Confirm missing-vs-zero distinction is real, not accidental ----------
# For one sample series, show whether gaps and zeros look genuinely different
sample_id = Y_product["unique_id"].iloc[0]
sample_series = (
    Y_product[Y_product["unique_id"] == sample_id]
    .set_index("ds")["y"]
    .reindex(full_calendar)
)
print(f"\nSample series: {sample_id}")
print("True gaps (no row at all):", sample_series.isna().sum())
print("Recorded zeros:", (sample_series == 0).sum())
print("Recorded nonzero sales:", (sample_series > 0).sum())
</pool_a>

<pool_b>
# ---------- Final rearesults = {}  # collects True/False per check for the final summary

# ---------------------------------------------------------------
# 1 & 2. Column names check
# ---------------------------------------------------------------
expected_cols = ["unique_id", "ds", "y"]
actual_cols = list(Y_product.columns)
results["columns_match"] = actual_cols == expected_cols
print("Y_product columns:", actual_cols, "-> expected:", expected_cols)

# ---------------------------------------------------------------
# 3. Dtype checks: ds must be datetime, y must be numeric
# ---------------------------------------------------------------
results["ds_is_datetime"] = pd.api.types.is_datetime64_any_dtype(Y_product["ds"])
results["y_is_numeric"] = pd.api.types.is_numeric_dtype(Y_product["y"])
print("ds dtype:", Y_product["ds"].dtype, "| y dtype:", Y_product["y"].dtype)

# ---------------------------------------------------------------
# 4. Every ds value must fall on a Monday (weekday() == 0)
# ---------------------------------------------------------------
non_monday = Y_product.loc[Y_product["ds"].dt.weekday != 0]
results["all_dates_are_monday"] = len(non_monday) == 0
print("Non-Monday rows found:", len(non_monday))

# ---------------------------------------------------------------
# 5. No duplicate (unique_id, ds) pairs
# ---------------------------------------------------------------
dupe_count = Y_product.duplicated(subset=["unique_id", "ds"]).sum()
results["no_duplicate_pairs"] = dupe_count == 0
print("Duplicate (unique_id, ds) pairs:", dupe_count)

# ---------------------------------------------------------------
# 6. Inspect S_product's structure before assuming anything about it
#    (the knowledge pool does not specify its exact shape)
# ---------------------------------------------------------------
print("\nS_product type:", type(S_product))
print("S_product index sample:", list(S_product.index[:5]))
print("S_product columns sample:", list(S_product.columns[:5]))

# Determine where the bottom-level identifiers live: if "unique_id" is a
# column, use it; otherwise assume the row/column index holds them, and
# the bottom-level count equals the number of columns (per the earlier
# established convention: S's columns = bottom-level series).
if "unique_id" in S_product.columns:
    bottom_ids_from_S = set(S_product["unique_id"])
else:
    bottom_ids_from_S = set(S_product.columns)

n_bottom = len(bottom_ids_from_S)
results["bottom_level_count_is_222"] = n_bottom == 222
print("Bottom-level count implied by S_product:", n_bottom, "(expected 222)")

# ---------------------------------------------------------------
# 7 & 8. Total unique_ids in Y_product, and whether S_product's
#    bottom-level ids are all present inside Y_product
# ---------------------------------------------------------------
y_product_ids = set(Y_product["unique_id"].unique())
print("\nTotal distinct unique_ids in Y_product (all hierarchy levels):", len(y_product_ids))
print("NOTE: this total includes Total/Category/Brand rollups, not just the 222 bottom SKUs.")

missing_from_Y = bottom_ids_from_S - y_product_ids
results["Y_product_matches_S_bottom"] = len(missing_from_Y) == 0
print("Bottom-level ids missing from Y_product:", len(missing_from_Y))

# ---------------------------------------------------------------
# 9. Per-series calendar check: gaps vs. zero-sale weeks, kept SEPARATE
#    and NOT filled or imputed, per FPP3 §13.9 and your Step 4.1 decision
# ---------------------------------------------------------------
gap_report = []
for uid, group in Y_product.groupby("unique_id"):
    start, end = group["ds"].min(), group["ds"].max()
    expected_weeks = pd.date_range(start, end, freq="W-MON")
    observed_weeks = set(group["ds"])
    missing_weeks = len(set(expected_weeks) - observed_weeks)     # true gaps — no row at all
    zero_weeks = (group["y"] == 0).sum()                          # recorded zeros — a real observation
    gap_report.append({
        "unique_id": uid,
        "n_expected_weeks": len(expected_weeks),
        "n_observed_rows": len(group),
        "missing_weeks": missing_weeks,
        "zero_sale_weeks": zero_weeks,
    })
gap_report = pd.DataFrame(gap_report)

total_missing_weeks = gap_report["missing_weeks"].sum()
total_zero_weeks = gap_report["zero_sale_weeks"].sum()
print(f"\nTotal true gap-weeks across all series (NOT filled): {total_missing_weeks}")
print(f"Total recorded zero-sale weeks across all series: {total_zero_weeks}")
print("Series with the most gaps (top 5):")
print(gap_report.sort_values("missing_weeks", ascending=False).head(5))

# ---------------------------------------------------------------
# 10. No NaN values in y
# ---------------------------------------------------------------
nan_count = Y_product["y"].isna().sum()
results["no_nan_in_y"] = nan_count == 0
print("\nNaN values in y:", nan_count)

# ---------------------------------------------------------------
# 11. Date range / series length summary
# ---------------------------------------------------------------
print("\nOverall ds range:", Y_product["ds"].min(), "to", Y_product["ds"].max())
print("Per-series row-count summary:")
print(Y_product.groupby("unique_id").size().describe())

# ---------------------------------------------------------------
# Final pass/fail summary
# ---------------------------------------------------------------
print("\n=== STEP 2.1 VALIDATION SUMMARY ===")
for check, passed in results.items():
    print(f"{'PASS' if passed else 'FAIL'} — {check}")

overall = all(results.values())
print(f"\nOVERALL: {'PASS' if overall else 'FAIL'}")
print("Reminder: gaps and zero-sale weeks were only counted and reported above — nothing was filled or imputed.")
diness check ----------
print("\nY_product is ready for StatsForecast:", not Y_product[["unique_id", "ds", "y"]].isna().any().any())

</pool_b>

<task>
Merge Pool A and Pool B into a single document in which every unique piece of information from either pool appears exactly once.
</task>

<merge_rules>
1. Deduplicate, don't drop. Where both pools say the same thing, keep the more complete or more precise version. Where one pool has a detail the other lacks, carry it over.

2. Preserve verbatim:
   - all code blocks
   - all FPP3 section numbers
   - all direct FPP3 quotes, e.g. "known to be biased", "lacks a statistical foundation for computing prediction intervals", MAPE "infinite or undefined if y_t=0"
   Do not silently edit code. If you spot a bug or questionable logic (e.g. the MASE function computing the naive scale across all series pooled rather than per series), leave the code as-is and add a clearly labeled "Reviewer note" beneath it.

3. Keep provenance labels intact. Every claim should stay tagged as one of:
   (a) verified from FPP3
   (b) verified from StatsForecast/Nixtla docs
   (c) engineering judgement, not from either source
   Never upgrade (c) to (a) or (b).

4. Model inventory:
   - Use Pool B's role-based grouping as the structure.
   - Fill it with every model from Pool A's flat table, including the reconciliation methods and the StatsForecast-only models with no FPP3 coverage (CrostonOptimized, CrostonSBA, TSB, ADIDA, IMAPA, AutoCES, the Theta family, MFLES, TBATS, GARCH/ARCH, WindowAverage variants).
   - Put reconciliation (Bottom-up, Top-down, Middle-out, MinT) in its own group, marked "Belongs to F3, not F1 Step 2.3", with Pool B's reasoning.
   - Mark the subset of models that are actual candidates for Step 2.3.

5. Checklist mapping:
   - Combine Pool B's two mapping tables into one table with three columns: checklist item, primary step, and "spans" notes.
   - Keep the observation that Step 2.5 carries the most checklist weight, and that 2.6 maps to nothing new.

6. The gap:
   - Keep Pool B's flag that no step explicitly compares candidate scores and discards the losers.
   - Then add a short proposed step "2.5b — Select winner per group" with minimal code: compare cross-validation MASE per model per group, keep the best, and pull its 13-week forecast.
   - Label this step clearly as "New — not in either source pool".

7. Resolve wording conflicts:
   - Pool B uses "my situation" and "flagged"; Pool A uses "your situation" and "flagged per your instruction". Normalize to one voice ("your project").
   - If the pools genuinely disagree on substance, show both versions side by side and say which you consider correct and why. Do not pick one silently.

8. Keep the sourcing caveats from Pool A:
   - the uploaded PDF wasn't available, so the public FPP3 was used
   - FPP3 has no dedicated intermittent-demand chapter
</merge_rules>

<output_structure>
0. Sources and caveats
1. Model inventory (grouped by role; Step 2.3 candidates marked; reconciliation set apart)
2. Step-by-step guide F1 / Hierarchy A: 2.1 → 2.2 → 2.3 → 2.4 → 2.5 → 2.5b (new) → 2.6
3. Checklist with the unified step-mapping table
4. Common pitfalls (union of both pools)
5. Merge log: a short list of
   - every place you deduplicated, with which version you kept
   - every place you resolved a conflict
   - every reviewer note you added
</output_structure>

<constraints>
- Do not invent FPP3 section numbers, quotes or StatsForecast parameters. If something is uncertain, say so.
- Use Markdown tables for model lists and mappings, and fenced Python blocks for code.
- Before finishing, check that every model, code block, checklist item, pitfall and caveat from both pools appears in your output. List anything you intentionally left out, with the reason.
</constraints>